# DLP Insider Threat Detection — Full Pipeline

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Get your Kaggle API key: kaggle.com → Account → API → *Create New Token* → downloads `kaggle.json`
3. Run all cells top to bottom

| Step | What happens |
|------|--------------|
| 1 | Clone repo from GitHub |
| 2 | Install dependencies |
| 3 | Upload Kaggle credentials |
| 4 | Download CERT dataset from Kaggle |
| 5 | Configure paths |
| 6 | Clean all raw data |
| 7 | Train Isolation Forest (baseline) |
| 8 | Visualize Isolation Forest results |
| 9 | Train LSTM Autoencoder (GPU) |
| 10 | Visualize LSTM results |
| 11 | Ground truth evaluation (CERT r4.2) |
| 12 | Launch monitoring dashboard |

## Step 1 — Clone repo from GitHub

In [ ]:
import os

REPO_URL = 'https://github.com/Taha-coder-star/DLP-PROJECt.git'
REPO_DIR = '/content/dlp-project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

## Step 2 — Install dependencies

In [ ]:
!pip install -r colab/requirements-colab.txt kaggle -q
print('Done.')

## Step 3 — Upload Kaggle credentials
Go to **kaggle.com → Account → API → Create New Token** to download `kaggle.json`, then run this cell and upload it.

In [ ]:
from google.colab import files

print('Upload your kaggle.json file:')
uploaded = files.upload()  # select kaggle.json from your computer

import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials configured.')

## Step 4 — Download CERT dataset from Kaggle

In [ ]:
KAGGLE_DATASET = 'mrajaxnp/cert-insider-threat-detection-research'
ARCHIVE_DIR    = '/content/dlp-data/archive'

import os, zipfile
from pathlib import Path
os.makedirs(ARCHIVE_DIR, exist_ok=True)

REQUIRED_FILES = [
    'email.csv', 'psychometric.csv', 'logon.csv',
    'device.csv', 'file.csv', 'decoy_file.csv', 'users.csv',
]

archive = Path(ARCHIVE_DIR)

for fname in REQUIRED_FILES:
    dest     = archive / fname
    zip_path = archive / (fname + '.zip')

    if zip_path.exists() and not dest.exists():
        print(f'  Unzipping {zip_path.name} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(ARCHIVE_DIR)
        zip_path.unlink()

    if dest.exists():
        print(f'  {fname} already present.')
        continue

    print(f'  Downloading {fname} ...')
    !kaggle datasets download -d {KAGGLE_DATASET} -f {fname} -p {ARCHIVE_DIR} --force
    if zip_path.exists():
        print(f'  Unzipping {zip_path.name} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(ARCHIVE_DIR)
        zip_path.unlink()

found   = sorted(archive.glob('*.csv'))
missing = set(REQUIRED_FILES) - {f.name for f in found}
print(f'\nFound {len(found)} CSV files:')
for f in found:
    print(f'  {f.name:40s} {f.stat().st_size / 1_048_576:8.1f} MB')
if missing:
    print('WARNING: missing files:', missing)
else:
    print('All required files present.')

### Step 4b — Download CERT answers file (ground truth)
The answers file is not on Kaggle — upload `insiders.csv` from the CERT r4.2 download package.
If you don't have it, skip this cell and skip Step 11.

In [ ]:
from google.colab import files
from pathlib import Path

ANSWERS_DIR = Path('/content/dlp-data/archive/answers/answers')
ANSWERS_DIR.mkdir(parents=True, exist_ok=True)

print('Upload insiders.csv from the CERT r4.2 answers folder:')
uploaded = files.upload()   # select insiders.csv

import shutil
for fname in uploaded:
    shutil.move(fname, str(ANSWERS_DIR / fname))
    print(f'Saved to {ANSWERS_DIR / fname}')

## Step 5 — Configure paths

In [ ]:
import os, sys
from pathlib import Path

os.environ['DLP_ROOT'] = '/content/dlp-data'

root = Path(os.environ['DLP_ROOT'])
for d in ['archive', 'cleaned', 'models', 'plots']:
    (root / d).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, '/content/dlp-project')
sys.path.insert(0, '/content/dlp-project/colab')

print('DLP_ROOT :', os.environ['DLP_ROOT'])
import torch
print('GPU      :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT available — go to Runtime > Change runtime type > T4 GPU')

## Step 6 — Clean all raw data
> Takes ~10–20 min (processes ~10M rows across all files).

In [ ]:
!python /content/dlp-project/scripts/clean_cert_email_data.py

In [ ]:
import pandas as pd

df = pd.read_csv('/content/dlp-data/cleaned/email_user_daily_with_psychometric.csv')
print(f'Daily feature matrix: {len(df):,} rows | {df["user"].nunique()} users')
print(f'Date range: {df["email_day"].min()} → {df["email_day"].max()}')
print('\nSplit distribution:')
print(df['dataset_split'].value_counts().to_string())

## Step 7 — Train Isolation Forest (baseline)

In [ ]:
!python /content/dlp-project/colab/train_isolation_forest_cert.py

In [ ]:
import json
with open('/content/dlp-data/models/isolation_forest_summary.json') as f:
    if_summary = json.load(f)

print('=== Isolation Forest Summary ===')
for k, v in if_summary.items():
    if k not in ('top_anomalies', 'top_feature_columns'):
        print(f'  {k}: {v}')
print('\nTop 10 anomalies:')
for a in if_summary['top_anomalies'][:10]:
    print(f'  {a["user"]}  {a["email_day"]}  score={a["iforest_score"]:.4f}  [{a["risk_severity"]}]')

## Step 8 — Visualize Isolation Forest results

In [ ]:
!python /content/dlp-project/colab/visualize_isolation_forest_cert.py

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

for img in sorted(Path('/content/dlp-data/plots').glob('iforest_*.png')):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(img))
    ax.axis('off')
    ax.set_title(img.stem)
    plt.tight_layout()
    plt.show()

## Step 9 — Train LSTM Autoencoder (GPU)
> Single global model trained on all users. Takes ~10–20 min on T4 GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU found! Go to Runtime > Change runtime type > T4 GPU and re-run from Step 5.'
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
!python /content/dlp-project/colab/train_lstm_autoencoder_cert.py

In [ ]:
!python /content/dlp-project/colab/inference_lstm_autoencoder_cert.py

In [ ]:
with open('/content/dlp-data/models/lstm_autoencoder_summary.json') as f:
    lstm_summary = json.load(f)

print('=== LSTM Autoencoder Summary ===')
for k, v in lstm_summary.items():
    if k != 'top_anomalies':
        print(f'  {k}: {v}')
print('\nTop 10 anomalies:')
for a in lstm_summary['top_anomalies'][:10]:
    print(f'  {a["user"]}  {a["email_day"]}  score={a["lstm_score"]:.4f}  [{a["lstm_risk_severity"]}]')

## Step 10 — Visualize LSTM results

In [ ]:
!python /content/dlp-project/colab/visualize_lstm_autoencoder_cert.py

for img in sorted(Path('/content/dlp-data/plots').glob('lstm_*.png')):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(img))
    ax.axis('off')
    ax.set_title(img.stem)
    plt.tight_layout()
    plt.show()

## Step 11 — Ground Truth Evaluation (CERT r4.2)
Evaluates both models against the 70 known insider users.
> Skip if you did not upload `insiders.csv` in Step 4b.

In [ ]:
from pathlib import Path

ANSWERS = Path('/content/dlp-data/archive/answers/answers/insiders.csv')

if not ANSWERS.exists():
    print('insiders.csv not found — skipping evaluation. Upload it in Step 4b and rerun.')
else:
    !python /content/dlp-project/scripts/evaluate_models.py --answers {ANSWERS} --split test

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

EVAL_PATH = Path('/content/dlp-data/models/evaluation_report.json')
if not EVAL_PATH.exists():
    print('Run Step 11 first.')
else:
    with open(EVAL_PATH) as f:
        report = json.load(f)

    sections = {
        'IF — Day (test)':   report.get('if_day_test',  {}),
        'IF — User (all)':   report.get('if_user_all',  {}),
        'LSTM — Day (test)': report.get('lstm_day_test',{}),
        'LSTM — User (all)': report.get('lstm_user_all',{}),
    }
    rows = [{'Model': k, **{m: v.get(m,'-') for m in ['roc_auc','avg_precision','precision','recall','f1','tp','fp','fn']}}
            for k, v in sections.items()]
    print(pd.DataFrame(rows).set_index('Model').to_string())

    # ROC curves
    ANSWERS = Path('/content/dlp-data/archive/answers/answers/insiders.csv')
    if ANSWERS.exists():
        ins = pd.read_csv(ANSWERS)
        ins = ins[ins['dataset'] == 4.2].copy()
        ins['start'] = pd.to_datetime(ins['start']).dt.normalize()
        ins['end']   = pd.to_datetime(ins['end']).dt.normalize()
        rows_gt = []
        for _, r in ins.iterrows():
            for d in pd.date_range(r['start'], r['end'], freq='D'):
                rows_gt.append({'user': r['user'], 'email_day': d, 'is_insider': 1})
        day_labels = pd.DataFrame(rows_gt).drop_duplicates()

        if_df   = pd.read_csv('/content/dlp-data/cleaned/email_user_daily_scored.csv')
        lstm_df = pd.read_csv('/content/dlp-data/cleaned/email_user_daily_lstm_scored.csv')
        if_df['email_day']   = pd.to_datetime(if_df['email_day']).dt.normalize()
        lstm_df['email_day'] = pd.to_datetime(lstm_df['email_day']).dt.normalize()

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))

        for ax, (df, score_col, label, color) in zip(axes, [
            (if_df[if_df['dataset_split']=='test'],   'iforest_score', 'Isolation Forest', '#4C9BE8'),
            (lstm_df[(lstm_df['dataset_split']=='test') & (lstm_df['lstm_risk_severity']!='undetermined')],
             'lstm_score', 'LSTM Autoencoder', '#E85454'),
        ]):
            merged = df.merge(day_labels, on=['user','email_day'], how='left')
            merged['is_insider'] = merged['is_insider'].fillna(0).astype(int)
            if merged['is_insider'].sum() > 0:
                fpr, tpr, _ = roc_curve(merged['is_insider'], merged[score_col].fillna(0))
                auc = roc_auc_score(merged['is_insider'], merged[score_col].fillna(0))
                ax.plot(fpr, tpr, lw=2, color=color, label=f'{label} (AUC={auc:.3f})')
            ax.plot([0,1],[0,1],'k--',lw=1)
            ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
            ax.set_title(f'{label} ROC — day level (test)')
            ax.legend()

        plt.tight_layout(); plt.show()

## Step 12 — Launch Monitoring Dashboard
Run the interactive 7-tab Streamlit app.

In [ ]:
!pip install streamlit pyngrok -q

import threading, time
from pyngrok import ngrok

ngrok.kill()

def run_streamlit():
    import subprocess
    subprocess.run([
        'streamlit', 'run', '/content/dlp-project/app/monitoring_app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)

tunnel = ngrok.connect(8501)
print('=' * 60)
print('Dashboard URL:', tunnel.public_url)
print('=' * 60)
print('Open the URL above in your browser.')
print('The app has 7 tabs:')
print('  1. Pipeline Overview       — data splits, model configs')
print('  2. Isolation Forest        — scores, severity, top users')
print('  3. LSTM Autoencoder        — scores, user timeline')
print('  4. Model Comparison        — IForest vs LSTM correlation')
print('  5. User Investigation      — per-user deep dive')
print('  6. Live Scoring            — score any row in real-time')
print('  7. Evaluation (Ground Truth) — precision, recall, ROC AUC')